# Multi-Objective Reinforcement Learning based on Decomposition with Policy Iteration

This notebook implements and evaluates our MORL personalization algorithm as described in Chapter 3 of our thesis. It trains the algorithm, discovers a Pareto-optimal set of policies, evaluates baseline methods, and conducts simulations with synthetic users. Thes results of the simulations are analyzed (in `simulation_analysis.py`) to assess the performance of our proposed method in comparison to other approaches.

**Author:** Shirley Li  
**Date:** July 2026

### Notebook Structure

The analysis follows these main phases:

1. **Data Preparation**: Load user interaction data and process it into MOMDP representation
2. **Environment Setup**: Initialize multi-objective gymnasium environment with estimated reward and transition dynamics
3. **Algorithm Training**: Train PIMORLD to discover Pareto-optimal policies
4. **Policy Analysis**: Extract and analyze discovered policies from Pareto archive
5. **Comparison Policies**: Implement multiple baseline policy methods for comparison
6. **Simulation**: Run simulations comparing all policy approaches
7. **Results Analysis**: Evaluation of metrics including adherence, retention, diversity, and dropout

### Parameters

Configuration variables (defined in `config.py`):
- **NUM_STATES**: Number of states
- **NUM_ACTIONS**: Total number of actions (challenge clusters × within-cluster selection strategies)
- **NUM_OBJECTIVES**: Number of reward objectives
- **gamma**: Discount factor
- **NUM_USERS**: Number of users (simulations) to run
- **NUM_TIMESTEPS**: Timesteps per user simulation

In [1]:
import pandas as pd
import numpy as np
import os
import mo_gymnasium as mo_gym
from env.user_sim import UserSimEnv
import env.simulate as sim_utils
import seaborn as sns
import pandas as pd
import utils.utils as utils
import utils.process_data as dp
import morld.mp_selection as mp_selection 
from morld.pi_morld import PIMORLD
import pyperclip

import utils.mdp_utils as mdp_utils
import utils.plot_figures as pf
import pickle
from morld.mo_pi import MOPolicyIteration
import config

In [2]:
state_to_idx, idx_to_state = utils.build_state_space(config.NUM_VALS_PER_FEATURE)

## Data preparation

First we load the data and process the samples

In [ ]:
results_folder = '..\\results\\'
data_folder = '..\\data\\'

action_data = pd.read_csv(os.path.join(data_folder, 'challenge_info.csv'))

samples = pd.read_csv(os.path.join(data_folder, 'processed_samples_corrected_full.csv'))
model_name_suffix = ""

actions_clustered, _, _ = dp.cluster_actions(action_data, config.cluster_vars, num_clusters=config.NUM_CLUSTERS)

df, agency_df, initial_distribution, mapping = dp.process_samples(samples, actions_clustered, config.state_features, config.NUM_VALS_PER_FEATURE, config.action_col, config.cluster_col)

For the reward estimation of some objectives, we only use data from completed challenges, as only then the reward can be observed. Otherwise the reward is 0.

In [4]:
# In some cases, the challenge was completed, but the ratings were not given (i.e. they are 0 after preprocessing).
# These are filtered out for the reward estimation
filtered_df = df[~(df['completed'] == 1 & (df['difficulty'] == 0))]

# For the stochastic rewards for simulation, we only want to use the completed samples where the ratings are given (i.e. not 0 after preprocessing)
completed_df = df[df['completed'] == 1]
completed_df = completed_df[completed_df['likedness'] != 0]

print("Total number of samples:", len(df))
print("Number of filtered samples (no nans for completed):", len(filtered_df))
print("Number of completed samples:", len(df[df['completed'] == 1]))
print("Number of filtered completed samples (no nans for completed):", len(completed_df))

Total number of samples: 3471
Number of filtered samples (no nans for completed): 3451
Number of completed samples: 2349
Number of filtered completed samples (no nans for completed): 2329


### Estimate transition and reward functions

In [5]:
# Completion probabilities, rewards, and transition probabilities for the MOMDP
P_comp = mdp_utils.compute_completion_probabilities(df, config.NUM_STATES, config.NUM_ACTIONS, config.action_col, prior='action_only')
R = mdp_utils.compute_rewards(filtered_df, config.NUM_STATES, config.NUM_ACTIONS, config.NUM_OBJECTIVES, config.reward_cols, config.action_col, mapping)
P = mdp_utils.compute_transition_probabilities(df, config.NUM_STATES, config.NUM_ACTIONS, config.action_col)

print("P shape:", P.shape)  # Should be (nU, nA, nU)
print("R shape:", R.shape)  # Should be (nU, nA, nO)

# Estimate completion probabilities for agency condition for user choice simulation
P_comp_agency = mdp_utils.compute_completion_probabilities(agency_df, config.NUM_STATES, config.NUM_ACTIONS, config.action_col, prior='action_only')

P shape: (12, 6, 12)
R shape: (12, 6, 5)


For our simulations, we use reward distributions. 

In [6]:
lam = 1  # Laplace smoothing

R_probs_fun = mdp_utils.compute_reward_probabilities(completed_df, 'likedness', config.NUM_STATES, config.NUM_ACTIONS, config.action_col, lam=lam)
R_probs_pu = mdp_utils.compute_reward_probabilities(completed_df, 'usefulness', config.NUM_STATES, config.NUM_ACTIONS, config.action_col, lam=lam)
R_probs_return = mdp_utils.compute_reward_probabilities(df, 'PAY_next', config.NUM_STATES, config.NUM_ACTIONS, config.action_col, lam=lam)

R_probs = {'likedness': R_probs_fun, 'usefulness': R_probs_pu, 'PAY_next': R_probs_return}

### Challenge information
We need some information on the challenges for our simulation environment

In [7]:
challenges_per_cluster = [[i for i in range(len(actions_clustered)) if actions_clustered[config.cluster_col].iloc[i] == cluster_id] for cluster_id in range(config.NUM_CLUSTERS)]
challenge_info = actions_clustered.set_index('challenge_id').to_dict('records')

## Setup environment

In [8]:
env = mo_gym.make('user_env', num_actions=config.NUM_ACTIONS, 
                 num_objectives = config.NUM_OBJECTIVES, 
                 num_vals_per_feature=config.NUM_VALS_PER_FEATURE,
                 challenge_info=challenge_info, 
                 challenges_per_cluster=challenges_per_cluster,
                 num_categories=config.NUM_CATEGORIES,
                 transition_probs=P, 
                 completion_probs=P_comp,
                 reward_matrix=R, 
                 reward_probs=R_probs,
                 initial_distribution=initial_distribution,
                 mapping=mapping,
                 P_comp_agency=P_comp_agency,
                 agency_bias_params=config.agency_bias,
                 )

## Policy Iteration MORL/D algorithm
Setup the algorithm and train it to discover Pareto-optimal policies. The discovered policies are stored in a Pareto archive.

In [9]:
pop_size = 128
weights_init = "combined"
scalarization = "linear"
max_eval_iters = 1000
gamma = 0.7

model_name = f'pimorld_P={pop_size}_gamma={gamma}_MOT{model_name_suffix}'
filename = model_name
model_file = os.path.join(results_folder, model_name, filename)

# Initialize the PIMORLD algorithm
pi_morld = PIMORLD(env.unwrapped,
                   pop_size=pop_size, 
                   gamma=gamma,
                   max_eval_iters=max_eval_iters,
                   weights_init=weights_init,
                   scalarization=scalarization)

# Train the model
pi_morld.train(num_iterations=1000, max_resets=50)

Initial weights for agents:
[[0.         0.         0.         0.         1.        ]
 [0.         0.         0.         0.20348524 0.79651476]
 [0.         0.         0.         0.40281049 0.59718951]
 [0.         0.         0.         0.60108623 0.39891377]
 [0.         0.         0.         0.79960042 0.20039958]
 [0.         0.         0.         1.         0.        ]
 [0.         0.         0.25916169 0.50793705 0.23290126]
 [0.         0.         0.25991581 0.         0.74008419]
 [0.         0.         0.26397129 0.25308243 0.48294628]
 [0.         0.         0.49094124 0.50905876 0.        ]
 [0.         0.         0.49559295 0.25665379 0.24775326]
 [0.         0.         0.50599475 0.         0.49400525]
 [0.         0.         0.74025807 0.25974193 0.        ]
 [0.         0.         0.74849568 0.         0.25150432]
 [0.         0.         1.         0.         0.        ]
 [0.         0.2487372  0.49492772 0.         0.25633508]
 [0.         0.25373475 0.24844609 0.       

Save to file

In [10]:
save_path = os.path.join(results_folder, model_name, 'model')
save_file = "pimorld_trained.pkl"
save_to = os.path.join(save_path, save_file)
if not os.path.exists(save_path):
    os.makedirs(save_path)

with open(save_to, 'wb') as f:
    pickle.dump(pi_morld, f)

Inspect the discovered approximation set

In [11]:
print("Hypervolume of Pareto Archive: ", pi_morld.get_hv())
print("Number of solutions in Pareto Archive: ", len(pi_morld.pareto_archive.evaluations))
mp_selection.get_max_unique_actions(pi_morld.pareto_archive.individuals, config.NUM_STATES)

Hypervolume of Pareto Archive:  50.778459212276125
Number of solutions in Pareto Archive:  174


{'max_unique_actions': 6,
 'avg_unique_actions': 3.9166666666666665,
 'most_disputed_state': 10}

### Subset selection from Pareto archive
For our metapolicies explained in Chapter 3 of the thesis, we need to select a subset of policies from the discovered Pareto archive. For this we select a subset based on the Expected Utility Metric, and the one that performs the best on the expert-driven objectives.

#### User choice: subset selection based on EUM

In [12]:
rng = np.random.default_rng(66)
weight_samples = rng.dirichlet(np.ones(config.NUM_OBJECTIVES), size=10000)

policy_evals = np.array(pi_morld.pareto_archive.evaluations)
eum = mp_selection.calculate_eum(policy_evals, weight_samples)
print("Expected Utility of Pareto Archive: ", eum)

selected_indices = mp_selection.select_policies(pi_morld.pareto_archive.individuals, policy_evals, mp_selection.score_eum, max_actions=3, weight_samples=weight_samples)
print(f"Best subset of policies has EUM: {mp_selection.calculate_eum(policy_evals[selected_indices], weight_samples)}")
top_n_individuals = np.array(pi_morld.pareto_archive.individuals)[selected_indices]
top_n_evaluations = np.array(pi_morld.pareto_archive.evaluations)[selected_indices]
selected_policies = np.array([ind['policy'] for ind in top_n_individuals])

policy_morl_raw = mp_selection.get_actions_per_state(top_n_individuals, config.NUM_STATES)

display(policy_morl_raw)

Expected Utility of Pareto Archive:  2.2042139309537347


Best subset of policies has EUM: 2.2030822946588646


{0: {np.int8(1), np.int8(4)},
 1: {np.int8(1), np.int8(4), np.int8(5)},
 2: {np.int8(1), np.int8(2), np.int8(5)},
 3: {np.int8(1), np.int8(4), np.int8(5)},
 4: {np.int8(1), np.int8(4), np.int8(5)},
 5: {np.int8(2), np.int8(4), np.int8(5)},
 6: {np.int8(1), np.int8(4), np.int8(5)},
 7: {np.int8(1), np.int8(4)},
 8: {np.int8(1), np.int8(4), np.int8(5)},
 9: {np.int8(1), np.int8(4), np.int8(5)},
 10: {np.int8(1), np.int8(4), np.int8(5)},
 11: {np.int8(1), np.int8(3), np.int8(4)}}

#### Expert-priority

In [13]:
best_expert_idx = np.argmax(np.sum(policy_evals[:, 2:], axis=1))  # expert-driven objectives are the last three objectives in the evaluation
best_expert_policy = pi_morld.pareto_archive.individuals[best_expert_idx]['policy']
print("Best expert-driven policy index: ", best_expert_idx)
print("Best expert-driven policy evaluation: ", policy_evals[best_expert_idx])
print(best_expert_policy)

Best expert-driven policy index:  4
Best expert-driven policy evaluation:  [1.62372996 1.38366041 1.96744118 2.54155351 3.33333121]
[4 4 5 5 4 4 4 4 4 4 3 3]


## Obtain comparison policies

In [14]:
# Equal weights for all objectives
equal_weights = np.ones(config.NUM_OBJECTIVES) / config.NUM_OBJECTIVES
solver_equal = MOPolicyIteration(P, R, equal_weights, gamma=gamma, scalarization=scalarization)
solver_equal.train()
if pi_morld.is_in_pareto_archive(solver_equal):
    print("Equal weights policy is in the Pareto archive.")
policy_equal = solver_equal.policy_table

# Weights based on correlations with completion
correlations = df[config.reward_cols + ['completed']].corr()['completed'][:-1]
weights_correlations = np.array(correlations) / np.sum(correlations)
print(correlations)
print(weights_correlations)
print("Weights based on correlations: ", weights_correlations)
solver_corr = MOPolicyIteration(P, R, weights_correlations, gamma=gamma, scalarization=scalarization)
solver_corr.train()
if pi_morld.is_in_pareto_archive(solver_corr):
    print("Correlation-based policy is in the Pareto archive.")
policy_corr = solver_corr.policy_table

print(policy_equal)
print(policy_corr)

Equal weights policy is in the Pareto archive.
r_likedness     0.832863
r_usefulness    0.800508
r_return        0.098211
r_adherence     1.000000
r_diversity    -0.005295
Name: completed, dtype: float64
[ 0.30549342  0.29362575  0.03602385  0.3667991  -0.00194213]
Weights based on correlations:  [ 0.30549342  0.29362575  0.03602385  0.3667991  -0.00194213]
Correlation-based policy is in the Pareto archive.
[4 4 5 4 4 5 4 4 4 4 4 3]
[1 1 2 1 1 5 1 4 1 4 2 1]


Format for table in thesis

In [15]:
joint_cluster_to_action = {
        0: ("D", 0),
        1: ("EU", 0),
        2: ("EUD", 0),
        3: ("D", 1),
        4: ("EU", 1),
        5: ("EUD", 1),
    }
def format_action(a):
    cluster, strategy = joint_cluster_to_action[a]
    return f"({cluster},{strategy})"

def format_action_set(actions):
    return r"\{" + ", ".join(format_action(a) for a in actions) + r"\}"

policy_df = pd.DataFrame({
    "State": [f"{idx_to_state[i]}" for i in range(config.NUM_STATES)],
    "B-Corr": [format_action(policy_corr[i]) for i in range(config.NUM_STATES)],
    "B-Eq": [format_action(policy_equal[i]) for i in range(config.NUM_STATES)],
    "MORL-EP": [format_action(best_expert_policy[i]) for i in range(config.NUM_STATES)],
    "MORL-UC": [format_action_set(policy_morl_raw[i]) for i in range(config.NUM_STATES)],
})

latex = policy_df.to_latex(index=False, escape=False)
pyperclip.copy(latex)

policy_df

,State,B-Corr,B-Eq,MORL-EP,MORL-UC
0,"(0, 0, 0)","(EU,0)","(EU,1)","(EU,1)","\{(EU,0), (EU,1)\}"
1,"(0, 0, 1)","(EU,0)","(EU,1)","(EU,1)","\{(EU,0), (EU,1), (EUD,1)\}"
2,"(0, 0, 2)","(EUD,0)","(EUD,1)","(EUD,1)","\{(EU,0), (EUD,0), (EUD,1)\}"
3,"(0, 1, 0)","(EU,0)","(EU,1)","(EUD,1)","\{(EU,0), (EU,1), (EUD,1)\}"
4,"(0, 1, 1)","(EU,0)","(EU,1)","(EU,1)","\{(EU,0), (EU,1), (EUD,1)\}"
5,"(0, 1, 2)","(EUD,1)","(EUD,1)","(EU,1)","\{(EUD,0), (EU,1), (EUD,1)\}"
6,"(1, 0, 0)","(EU,0)","(EU,1)","(EU,1)","\{(EU,0), (EU,1), (EUD,1)\}"
7,"(1, 0, 1)","(EU,1)","(EU,1)","(EU,1)","\{(EU,0), (EU,1)\}"
8,"(1, 0, 2)","(EU,0)","(EU,1)","(EU,1)","\{(EU,0), (EU,1), (EUD,1)\}"
9,"(1, 1, 0)","(EU,1)","(EU,1)","(EU,1)","\{(EU,0), (EU,1), (EUD,1)\}"


### Single-objective policies

In [16]:
# Policies that only optimize for one objective
single_objective_policies = {}
for i in range(config.NUM_OBJECTIVES):
    weights_single = np.zeros(config.NUM_OBJECTIVES)
    weights_single[i] = 1
    solver_single = MOPolicyIteration(P, R, weights_single, gamma=gamma, scalarization=scalarization)
    solver_single.train()
    if pi_morld.is_in_pareto_archive(solver_single):
        print(f"Single-objective policy for {config.reward_names[i]} is in the Pareto archive.")
    else:
        print(f"Single-objective policy for {config.reward_names[i]} is NOT in the Pareto archive.")
    policy_single = solver_single.policy_table
    single_objective_policies[f"SO-{config.reward_names[i]}"] = policy_single

Single-objective policy for Perceived Enjoyment is in the Pareto archive.
Single-objective policy for Perceived Usefulness is in the Pareto archive.
Single-objective policy for Return Willingness is in the Pareto archive.
Single-objective policy for Adherence is in the Pareto archive.
Single-objective policy for Diversity is NOT in the Pareto archive.


In [17]:
single_policies_df = pd.DataFrame({
    "State": [f"{idx_to_state[i]}" for i in range(config.NUM_STATES)],
    **{f"{config.reward_names[i]}": [format_action(policy_single[i]) for i in range(config.NUM_STATES)] for i, policy_single in enumerate(single_objective_policies.values())}
})
latex_single = single_policies_df.to_latex(index=False, escape=False)
pyperclip.copy(latex_single)

single_policies_df

,State,Perceived Enjoyment,Perceived Usefulness,Return Willingness,Adherence,Diversity
0,"(0, 0, 0)","(EU,0)","(EU,0)","(EU,1)","(EU,0)","(D,1)"
1,"(0, 0, 1)","(EU,1)","(EUD,0)","(EU,0)","(EU,0)","(D,1)"
2,"(0, 0, 2)","(EUD,0)","(EUD,0)","(EU,0)","(EUD,1)","(D,1)"
3,"(0, 1, 0)","(EU,0)","(EU,0)","(EUD,1)","(EU,0)","(EUD,1)"
4,"(0, 1, 1)","(EU,0)","(EU,0)","(EU,0)","(EU,0)","(EU,1)"
5,"(0, 1, 2)","(EUD,1)","(EUD,1)","(EUD,0)","(EUD,1)","(EU,1)"
6,"(1, 0, 0)","(EU,0)","(EU,0)","(EU,0)","(EU,0)","(D,1)"
7,"(1, 0, 1)","(EU,1)","(EU,1)","(D,0)","(EU,1)","(EU,1)"
8,"(1, 0, 2)","(EU,0)","(EU,0)","(D,0)","(EU,1)","(EUD,1)"
9,"(1, 1, 0)","(EU,1)","(EUD,1)","(EUD,1)","(EU,1)","(EU,1)"


## Run simulations
We first define the number of users and the number of timesteps to simulate

In [ ]:
NUM_USERS = config.NUM_USERS
NUM_TIMESTEPS = config.NUM_TIMESTEPS

save_path = os.path.join(results_folder, model_name, 'simulations')
save = False

#### Simulate comparison policies

In [ ]:
comparison_policies = []

# Random policy
simulation_random = sim_utils.simulate(env, NUM_USERS)
comparison_policies.append(simulation_random)

# Equal weights policy
simulation_equal = sim_utils.simulate(env, NUM_USERS, policy=policy_equal, policy_name="Equal Weights Policy")
comparison_policies.append(simulation_equal)

# Correlation-based policy
simulation_corr = sim_utils.simulate(env, NUM_USERS, policy=policy_corr, policy_name="Correlation-based Policy")
comparison_policies.append(simulation_corr)

# Single-objective policies
for policy_name, policy_single in single_objective_policies.items():
    simulation_single = sim_utils.simulate(env, NUM_USERS, policy=policy_single, policy_name=policy_name)
    comparison_policies.append(simulation_single)

#### Simulate metapolicies

In [ ]:
metapolicies = []

multi_policy_simulation_results_random = sim_utils.simulate_multiple_policies(env, NUM_USERS, selected_policies, top_n_evaluations, selection='most_likely', T=NUM_TIMESTEPS, policy_name="MORL-UC")
metapolicies.append(multi_policy_simulation_results_random)

multi_policy_simulation_results_expert = sim_utils.simulate(env, NUM_USERS, best_expert_policy, "MORL-EP", T=NUM_TIMESTEPS)
metapolicies.append(multi_policy_simulation_results_expert)

multi_policy_simulation_results_combined = sim_utils.simulate_multiple_policies(env, NUM_USERS, selected_policies, top_n_evaluations, selection='combined', T=NUM_TIMESTEPS, policy_name="MORL-UC+EP")
metapolicies.append(multi_policy_simulation_results_combined)

### Analyze and visualize simulations

In [ ]:
combined_df = pd.concat(comparison_policies + metapolicies, ignore_index=True)
pf.interactive_plot_objective(combined_df, config.reward_names)

In [ ]:
# save simulation results
if save:
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    combined_df.to_pickle(os.path.join(save_path, 'combined_simulation_results.pkl'))

#### Inspect adherence

In [ ]:
pf.plot_simulation(combined_df, 'num_completed')

#### Inspect number of users per state

In [ ]:
pf.plot_state_trajectory(combined_df, threshold=1)

#### Inspect diversity

In [ ]:
pf.plot_fraction_completed(combined_df, 4)

#### Analyze dropout / retention

In [ ]:
pf.plot_dropout(combined_df)

### Simulate multiple simulations to get more reliable estimates of dropout and motivation trajectory
In addition to the runs above, we want to get more reliable estimates of expected user retention and motivation trajectories. Therefore, we run multiple simulations for each policy and average the results. This part of the code produced Figure 4.4 in Chapter 4 of the thesis. The results are stored in `simulation_results_multiple_runs.pkl` and can be analyzed in `simulation_analysis.py`. Be aware that this part of the code takes a long time to run.

In [ ]:
num_runs = 20
num_users_per_run = 500

seed = 66 + NUM_USERS  

total_runs = num_runs * num_users_per_run

In [ ]:
comparison_policies_multi_run = []

# Random policy
simulation_random = sim_utils.simulate(env, total_runs, seed=seed)
comparison_policies_multi_run.append(simulation_random)

# Equal weights policy
simulation_equal = sim_utils.simulate(env, total_runs, seed=seed, policy=policy_equal, policy_name="Equal Weights Policy")
comparison_policies_multi_run.append(simulation_equal)

# Correlation-based policy
simulation_corr = sim_utils.simulate(env, total_runs, seed=seed, policy=policy_corr, policy_name="Correlation-based Policy")
comparison_policies_multi_run.append(simulation_corr)

# Single-objective policies
for policy_name, policy_single in single_objective_policies.items():
    simulation_single = sim_utils.simulate(env, total_runs, seed=seed, policy=policy_single, policy_name=policy_name)
    comparison_policies_multi_run.append(simulation_single)

In [ ]:
metapolicies_multi_run = []

multi_policy_simulation_results_random = sim_utils.simulate_multiple_policies(env, total_runs, selected_policies, top_n_evaluations, seed=seed, selection='most_likely', T=NUM_TIMESTEPS, policy_name="MORL-UC")
metapolicies_multi_run.append(multi_policy_simulation_results_random)

multi_policy_simulation_results_expert = sim_utils.simulate(env, total_runs, seed=seed, policy=best_expert_policy, policy_name="MORL-EP", T=NUM_TIMESTEPS)
metapolicies_multi_run.append(multi_policy_simulation_results_expert)

multi_policy_simulation_results_combined = sim_utils.simulate_multiple_policies(env, total_runs, selected_policies, top_n_evaluations, seed=seed, selection='combined', T=NUM_TIMESTEPS, policy_name="MORL-UC+EP")
metapolicies_multi_run.append(multi_policy_simulation_results_combined)

In [ ]:
combined_df_multi_run = pd.concat(comparison_policies_multi_run + metapolicies_multi_run, ignore_index=True)

if save:
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    combined_df_multi_run.to_parquet(os.path.join(save_path, 'combined_simulation_results_multi_run.parquet'))

In [ ]:
palette = sns.color_palette("muted")
sns.set_palette(palette)
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.8)


to_show = ["C-Corr", "C-Eq", "MORL-EP", "MORL-UC", "MORL-UC+EP", "Random"]
pf.plot_batched_dropout(combined_df_multi_run[combined_df_multi_run['policy'].isin(to_show)], batch_size=num_users_per_run, n_batches=num_runs, save_path=results_folder + "\\figures\\dropout_over_time.svg")

In [ ]:
pf.plot_batched_dropout(combined_df_multi_run, batch_size=num_users_per_run, n_batches=num_runs, save_path=results_folder + "\\figures\\extra\\dropout_over_time_so.svg")